In [3]:
# =============================================================================
# Project: Multimodal Sequential Recommendation
# Notebook: 02_Data_Statistics.ipynb
# Author: Manit Chitnukul
# Description:
# Calculates fundamental dataset statistics ( Sparsity, Scalle, Image Coverage).
# This metrics are curcial for the 'Data Description' section in the thesis.
# ============================================================================= 

import os
import gzip
import json 
from tqdm import tqdm 

# --- CONFIGURATION ---
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, 'data')

# Files (Raw Dataset)
REVIEW_FILE = os.path.join(DATA_DIR, 'reviews_Beauty_5.json.gz')
META_FILE = os.path.join(DATA_DIR, 'meta_Beauty.json.gz')

print(f" Analyzing Data at: {DATA_DIR}")



 Analyzing Data at: /Users/hector/Developments/IS_Multimodal_Recommendation/data


In [ ]:
def analyze_dataset_stats():
    print("Starting Data Statistice Anylysis...")

    # Use Sets for 0(1) uniqueness 
    users = set()
    items = set()
    interactions = 0

    # --- Scan Reviews (Count Users, Items, Interactions) ---
    print("Scanning Reviews (Counting interactions)...")

    if not os.path.exists(REVIEW_FILE):
        print(f"Error: Review file not found: {REVIEW_FILE}")
        return
    
    with gzip.open(REVIEW_FILE, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc="Processing Reviews"):
            try:
                line_data = json.loads(line.strip())

                # Extract IDs
                u_id = line_data.get('reviewerID')
                i_id = line_data.get('asin')

                if u_id and i_id:
                    users.add(u_id)
                    items.add(i_id)
                    interactions += 1
                
            except json.JSONDecodeError:
                continue

    # --- Scan Metadata (Check Image Coverage) ---
    print("\n Scanning Metadata (checking Image Coverage)...")

    items_with_images = set() # ASIN 's products with images

    if not os.path.exists(META_FILE):
        print(f" Error: Meta file not found: {META_FILE}")
        return
    
    with gzip.open(META_FILE, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc="Checking Metadata"):
            try:
                data  = json.loads(line.strip())
                asin = data.get('asin')

                # Filter: Only check items that actually exist in our review dataset (don't care about items with images but no one bought)
                if asin in items:
                    imgs = data.get('imageURL', []) or data.get('image', []) # Check both fields for image URLs

                    if isinstance(imgs, list) and len(imgs) > 0:
                        items_with_images.add(asin)
            
            except json.JSONDecodeError:
                continue

    # --- Calculate Metrics ---
    n_users = len(users)
    n_items = len(items)
    n_images = len(items_with_images)

    # Sparsity Calculation (Standard Formula in RecSys)
    # Sparsity = 1 - (Interactions / (Users * Items))
    matrix_size = n_users * n_items
    if matrix_size > 0:
        density = (interactions / matrix_size) * 100
        sparsity = 100 - density
    else:
        sparsity = 0
        density = 0
    
    img_coverage = (n_images / n_items) * 100 if n_items > 0 else 0

    # --- Report ---
    print("\n" + "="*50)
    print(" FINAL DATASET STATISTICS (Luxury Beauty)")
    print("="*50)
    print(f" Total Users:        {n_users:,}")
    print(f" Total Items:        {n_items:,}")
    print(f" Total Interactions: {interactions:,}")
    print("-" * 30)
    print(f" Data Sparsity:      {sparsity:.4f}%")
    print(f" Data Density:       {density:.4f}%")
    print("-" * 30)
    print(f" Items with Images:  {n_images:,}")
    print(f" Image Coverage:     {img_coverage:.2f}%")
    print("="*50)

    # Verification
    if img_coverage > 70:
        print(f"\n Status: Excellent! Dataset is ready for Multimodal Learning. {sparsity:.2f}% sparsity, {img_coverage:.2f}% image coverage.")
    elif img_coverage > 40:
        print(f"\n Status: Moderate. We may need to filter items without images later. {sparsity:.2f}% sparsity, {img_coverage:.2f}% image coverage.")
    else:
        print(f"\n Status: Low Coverage. Need to investigate why images are missing. {sparsity:.2f}% sparsity, {img_coverage:.2f}% image coverage.")

# Run the analysis
if __name__ == "__main__":
    analyze_dataset_stats() 